## SQL Injection

### Vad är SQL Injection?

SQL Injection är en attack där en användare lyckas skicka in egen SQL-kod genom ett inputfält.

Attackaren försöker få databasen att:
- läsa data
- ändra data
- ta bort data
- skapa nya användare
- köra egna SQL-kommandon

---

### Hur uppstår SQL Injection?

SQL Injection uppstår när användarinput sätts direkt in i en SQL-query.

####  Osäkert exempel

```python
search_string = input()

query = f"""
SELECT *
FROM users
WHERE name = '{search_string}'
"""
```

Här byggs SQL-queryn genom string concatenation.

---

#### Problem

Om användaren skriver:

```text
'; DROP TABLE users; --
```

blir queryn:

```sql
SELECT *
FROM users
WHERE name = '';
DROP TABLE users; --'
```

Nu kan databasen:
- avsluta första queryn
- köra attackarens query
- ignorera resten med `--`

---

### Viktiga symboler

#### Semicolon `;`

```sql
;
```

Används för att avsluta en SQL-query och börja en ny.

---

#### Kommentar `--`

```sql
--
```

Ignorerar resten av raden i SQL.

---

## Varför är detta farligt?

Attackaren kan göra samma operationer som i:
- SQL Server Management Studio
- databashanteringsverktyg

Exempel:

```sql
DROP TABLE users
DELETE FROM users
SELECT * FROM users
UPDATE users
```

---

### Unsafe Query Example

```python
unsafe_query = f"""
SELECT *
FROM airports
WHERE [Location served] LIKE '%""" + search_string + """%'
"""
```

Problem:
- användarinput sätts direkt in i SQL

---

### Parametrized Queries (Safe Solution)

Den säkra lösningen är parametriserade queries.

Istället för att klistra in användarinput direkt:
- används placeholders/parametrar
- data skickas separat från SQL-koden

---

### Säker Query

```python
query = text("""
SELECT *
FROM airports
WHERE [Location served] LIKE :search
""")
```

---

#### Execute with Parameters

```python
result = conn.execute(
    query,
    {"search": f"%{search_string}%"}
)
```

---

#### Hur fungerar det?

SQLAlchemy skickar:
- SQL-koden separat
- användardata separat

Databasen behandlar input som:
- DATA
- inte SQL-kod

---

### Viktig skillnad

#### ❌ Osäkert

```python
"..."+user_input+"..."
```

Användaren får kontroll över SQL-queryn.

---

## ✅ Säkert

```python
:name
```

och:

```python
{"name": user_input}
```

SQL-strukturen är låst och bara värden ändras.

---

#### Fördelar med Parametrized Queries

- skyddar mot SQL Injection
- säkrare kod
- bättre standard
- används i moderna system
- SQLAlchemy hanterar escaping automatiskt


### MARS (Multiple Active Result Sets)

#### Vad är MARS?

MARS står för:
```text
Multiple Active Result Sets
```

Det är en funktion i SQL Server som tillåter:
- flera SQL statements
- flera requests/resultat

på samma connection samtidigt.

```python
connect_args={"MARS_Connection": "Yes"}
```

---

#### Utan MARS

SQL Server tillåter normalt bara:
- en query åt gången

Om någon försöker göra:

```sql
SELECT * FROM users;
DROP TABLE users;
```

ignoreras ofta den andra queryn.

---

#### Med MARS aktiverat

Databasen tillåter:
- flera statements i samma request

Då kan SQL Injection fungera lättare.

---

### Viktig idé

MARS är egentligen inte farligt i sig.

Men:
- flera statements samtidigt
- kan göra SQL Injection värre


Moderna databaser försöker skydda mot SQL Injection automatiskt.

Men utvecklaren måste fortfarande:
- skriva säker kod
- använda parametrar
- aldrig concatenata user input i SQL

---

### Två sätt att använda parametrar

#### 1. Parametrar direkt i SQL (T-SQL)

```sql
SELECT *
FROM users
WHERE username = @username_input;
```

---

#### Hur fungerar det?

- SQL-queryn är statisk
- användardata skickas separat

Databasen behandlar:
- `@username_input`
som data
- inte SQL-kod

---

#### 2. Bound Parameters i SQLAlchemy

##### Query

```python
query = text("""
SELECT *
FROM users
WHERE FirstName LIKE :first_name
""")
```

---

##### Execute

```python
conn.execute(query, {
    "first_name": value
})
```

---

##### Hur fungerar det?

SQLAlchemy skickar:
- SQL-koden separat
- parametervärdet separat

Databasen vet då:
- vad som är SQL
- vad som är data

---

### Varför är detta säkert?

Även om användaren skriver:

```text
'; DROP TABLE users; --
```

kommer databasen behandla det som:
- vanlig text/data

inte som SQL-kod.

In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from urllib.parse import unquote
from getpass import getpass
server_name   = "localhost"
database_name = "everyloop"
user_name = "sa"
pwd = getpass("Password:")
connection_string = f"DRIVER=ODBC Driver 18 for SQL Server;SERVER={server_name};UID={user_name};PWD={pwd};DATABASE={database_name};TrustServerCertificate=yes"
url_string        = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

try:    
    engine = create_engine(url_string, connect_args={"MARS_Connection": "Yes"})
    with engine.connect() as connection:
        print(f'Successfully connected to {database_name}!')
except Exception as e:
    print('Error while connecting to database:\n')
    print(e)

Error while connecting to database:

(pyodbc.InterfaceError) ('28000', "[28000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Login failed for user 'sa'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Login failed for user 'sa'. (18456)")
(Background on this error at: https://sqlalche.me/e/20/rvf5)


In [ ]:
from sqlalchemy import text

search_string = input("Search airports: ")

query = text("""
SELECT TOP 10
    IATA,
    [Airport name] AS name,
    [Location served] AS location
FROM airports
WHERE [Location served] LIKE :search
""")

with engine.connect() as conn:

    result = conn.execute(
        query,
        {"search": f"%{search_string}%"}
    )

    print(f"{'IATA'.ljust(8)}{'Airport name'.ljust(50)}{'Location'}")

    for airport in result:
        print(
            f"{str(airport.IATA).ljust(8)}"
            f"{str(airport.name).ljust(50)}"
            f"{airport.location}"
        )